## Combining Legislator info with Roll Call Voting

### 1. Legislaor Info Extraction

In [ ]:
import requests
import pandas as pd
import time

BASE_URL = "https://malegislature.gov/api/GeneralCourts"

# Retrieve the list of member codes for each session
def get_member_codes(session):
    url = f"{BASE_URL}/{session}/LegislativeMembers"
    resp = requests.get(url)
    if resp.status_code != 200:
        print(f"Failed to fetch member list for session {session}")
        return []
    data = resp.json()
    return [item["MemberCode"] for item in data]


# Use the member code to obtain detailed information about the legislator.
def get_member_details(session, member_code):
    url = f"{BASE_URL}/{session}/LegislativeMembers/{member_code}"
    resp = requests.get(url)
    if resp.status_code != 200:
        print(f"Failed to fetch details for {member_code} in session {session}")
        return None

    data = resp.json()

    fields = [
        "Name",
        "GeneralCourtNumber",
        "LeadershipPosition",
        "Branch",
        "MemberCode",
        "District",
        "Party",
        "EmailAddress",
        "RoomNumber",
        "PhoneNumber",
    ]

    result = {f: data.get(f) for f in fields}
    return result


# Iterate through sessions 183–194
all_records = []

for session in range(183, 195): 
    print(f"\n=== Processing Session {session} ===")
    codes = get_member_codes(session)
    print(f"Found {len(codes)} member codes")

    for code in codes:
        details = get_member_details(session, code)
        if details:
            all_records.append(details)

        time.sleep(0.2)  

# save to CSV 
df = pd.DataFrame(all_records)
df.to_csv("all_sessions_legislators.csv", index=False)

print("\nSaved as all_sessions_legislators.csv")
print(df.head())



=== Processing Session 183 ===
Found 206 member codes
Failed to fetch details for A&F7 in session 183

=== Processing Session 184 ===
Found 211 member codes
Failed to fetch details for A&F7 in session 184

=== Processing Session 185 ===
Found 215 member codes
Failed to fetch details for A&F7 in session 185

=== Processing Session 186 ===
Found 204 member codes

=== Processing Session 187 ===
Found 204 member codes

=== Processing Session 188 ===
Found 211 member codes

=== Processing Session 189 ===
Found 208 member codes

=== Processing Session 190 ===
Found 208 member codes

=== Processing Session 191 ===
Found 205 member codes

=== Processing Session 192 ===
Found 202 member codes

=== Processing Session 193 ===
Found 205 member codes

=== Processing Session 194 ===
Found 202 member codes

Saved as all_sessions_legislators.csv
                Name  GeneralCourtNumber LeadershipPosition  Branch  \
0   Brian S. Dempsey                 183               None   House   
1        Cory A

In [ ]:
df = pd.read_csv("all_sessions_legislators.csv")

df["MemberCode"] = df["MemberCode"].astype(str)

df_house = df[~df["MemberCode"].str.endswith("0")]

count_all = df_house.groupby("GeneralCourtNumber")["MemberCode"].nunique().reset_index()
count_all.columns = ["Session", "HouseLegislators"]
print(count_all)

print(df_house.shape)
df_house.head()



    Session  HouseLegislators
0       183               164
1       184               169
2       185               170
3       186               162
4       187               164
5       188               169
6       189               166
7       190               164
8       191               164
9       192               162
10      193               165
11      194               163
(1982, 10)


,Name,GeneralCourtNumber,LeadershipPosition,Branch,MemberCode,District,Party,EmailAddress,RoomNumber,PhoneNumber
0,Brian S. Dempsey,183,NaN,House,BSD1,NaN,Democrat,NaN,NaN,NaN
1,Cory Atkins,183,NaN,House,C A1,NaN,Democrat,NaN,NaN,NaN
2,Carol A. Donovan,183,NaN,House,CADO1,NaN,Democrat,NaN,NaN,NaN
4,Charles A. Murphy,183,NaN,House,CAM1,NaN,Democrat,NaN,NaN,NaN
5,Cheryl A. Rivera,183,NaN,House,CAR1,NaN,Democrat,NaN,NaN,NaN


In [49]:
df_house.to_csv("all_sessions_legislators_house.csv", index=False)

#### 2. Merged House Roll Call Data with Legislators info 

In [ ]:
import pandas as pd
import re
from fuzzywuzzy import fuzz, process

def normalize_parts(name):
    """Extract all letter segments"""
    return re.findall(r"[A-Za-z]+", name)


def match_single_name(rollcall_name, member_df, session=None):
    """
    Strategy:
    1. Extract letter segments
    2. Use the longest letter segment as the matching key (typically the last name)
    3. Search legislators for all individuals containing the longest segment
    4. If candidates == 1 → Match successful
    5. If candidates >= 2 → No match (to avoid confusion)
    """
    if pd.isna(rollcall_name):
        return None, 0

    clean = str(rollcall_name).strip()
    clean = re.sub(r'^(Mr\.|Mrs\.|Ms\.)\s*', '', clean)
    clean = re.sub(r'--+', '', clean).strip()

    parts = normalize_parts(clean)
    if not parts:
        return None, 0

    # longest segment
    longest = max(parts, key=len).lower()

    candidates = member_df[
        member_df["Name"].str.lower().str.contains(longest, na=False)
    ]

    if len(candidates) == 1:
        return candidates.iloc[0], 100
    else:
        return None, 0


In [ ]:
def match_rollcall_to_members(rollcall_df, member_df):
    
    results = []
    unmatched = []

    for session in rollcall_df['GeneralCourtNumber'].unique():
        print(f"\nProcessing Session {session}...")
        
        session_rollcalls = rollcall_df[rollcall_df['GeneralCourtNumber'] == session]
        session_members = member_df[member_df['GeneralCourtNumber'] == int(session)]
        
        print(f"  Roll Call records: {len(session_rollcalls)}, legislators list: {len(session_members)}")
        
        unique_names = session_rollcalls['RollCallName'].unique()
        name_mapping = {}  
        
        for rollcall_name in unique_names:
            member_row, score = match_single_name(rollcall_name, session_members, session)
            name_mapping[rollcall_name] = (member_row, score) 
        
        for idx, row in session_rollcalls.iterrows():
            member_row, score = name_mapping[row['RollCallName']]   # 正确解包
            
            result = {
                    # Roll Call
                    'GeneralCourtNumber': row['GeneralCourtNumber'],
                    'RollCallNumber': row['RollcallNumber'],
                    'Date': row['Date'],
                    'TitleText': row['TitleText'],
                    'BillNumber': row['BillNumber'],
                    'AmendmentNumber': row['AmendmentNumber'],
                    'Vote': row['Vote'],
                    "RollCallName": row["RollCallName"],
                    'YeasCount': row['YeasCount'],
                    'NaysCount': row['NaysCount'],
                    'NVCount': row['NVCount'],
                    
                    # legis
                    'MatchedName': member_row.get('Name') if member_row is not None else None,
                    'MatchedMemberCode': member_row.get('MemberCode') if member_row is not None else None,
                    'LeadershipPosition': member_row.get('LeadershipPosition') if member_row is not None else None,
                    'Branch': member_row.get('Branch') if member_row is not None else None,
                    'District': member_row.get('District') if member_row is not None else None,
                    'Party': member_row.get('Party') if member_row is not None else None,
                    'EmailAddress': member_row.get('EmailAddress') if member_row is not None else None,
                    'RoomNumber': member_row.get('RoomNumber') if member_row is not None else None,
                    'PhoneNumber': member_row.get('PhoneNumber') if member_row is not None else None,    
                
            }
                
            results.append(result)
            
            if member_row is None:
                unmatched.append({
                    "session": session,
                    "rollcall_name": row["RollCallName"]
                })
        
        # Count the number of successful matches
        matched_count = sum(1 for n in unique_names if name_mapping[n][0] is not None)
        print(f"  Matched successfully: {matched_count}/{len(unique_names)} unique names")
    
    result_df = pd.DataFrame(results)
    
    if unmatched:
        unmatched_unique = pd.DataFrame(unmatched).drop_duplicates()
        print(f"\nUNKNOWN name ({len(unmatched_unique)}):")
        for _, row in unmatched_unique.iterrows():
            print(f"  Session {row['session']}: {row['rollcall_name']}")
    
    return result_df
        
        

##### 183-185

In [ ]:
rollcall_df = pd.read_csv("rollcalls_183_185.csv")
member_df = df_house

# Execute Match
matched_df = match_rollcall_to_members(rollcall_df, member_df)

#matched_df.to_csv("rollcalls_matched.csv", index=False)


Processing Session 183...
  Roll Call records: 75524, legislators list: 164
  Matched successfully: 145/161 unique names

Processing Session 184...
  Roll Call records: 77272, legislators list: 169
  Matched successfully: 148/174 unique names

Processing Session 185...
  Roll Call records: 94698, legislators list: 170
  Matched successfully: 154/176 unique names

UNKNOWN name (64):
  Session 183: Mr. Speaker
  Session 183: Murphy, K.J.
  Session 183: Golden B.P.
  Session 183: Golden T.A.
  Session 183: Bradley
  Session 183: Walsh, M.J.
  Session 183: Walsh S.
  Session 183: Rush
  Session 183: George
  Session 183: Hill
  Session 183: Stanley H.L.
  Session 183: Stanley T.M.
  Session 183: Murphy, C.A.
  Session 183: Sullivan, D.B.
  Session 183: Murphy, J.M.
  Session 183: Sullivan, J.C.
  Session 184: Mr. Speaker
  Session 184: Golden T.A.
  Session 184: Patrick
  Session 184: Bradley
  Session 184: Walsh, M.J.
  Session 184: Walsh S.
  Session 184: Keenan, Daniel
  Session 184: K

In [128]:
matched_df.head()

,GeneralCourtNumber,RollCallNumber,Date,TitleText,BillNumber,AmendmentNumber,Vote,RollCallName,YeasCount,NaysCount,NVCount,MatchedName,MatchedMemberCode,LeadershipPosition,Branch,District,Party,EmailAddress,RoomNumber,PhoneNumber
0,183,1,NaN,QUORUM,NaN,NaN,Y,Mr. Speaker,151,0,9,None,None,NaN,None,NaN,None,NaN,NaN,NaN
1,183,1,NaN,QUORUM,NaN,NaN,Y,Fresolo,151,0,9,John P. Fresolo,JPF1,NaN,House,NaN,Democrat,NaN,NaN,NaN
2,183,1,NaN,QUORUM,NaN,NaN,Y,"Murphy, K.J.",151,0,9,None,None,NaN,None,NaN,None,NaN,NaN,NaN
3,183,1,NaN,QUORUM,NaN,NaN,Y,Swan,151,0,9,Benjamin Swan,B S1,NaN,House,NaN,Democrat,NaN,NaN,NaN
4,183,1,NaN,QUORUM,NaN,NaN,Y,Asselin,151,0,9,Christopher P. Asselin,CPA1,NaN,House,NaN,Democrat,NaN,NaN,NaN


In [129]:
matched_df["GeneralCourtNumber"] = matched_df["GeneralCourtNumber"].astype(int)

leg_count = (
    matched_df.groupby("GeneralCourtNumber")["RollCallName"]
              .nunique()
              .reset_index(name="LegislatorCount")
              .sort_values("GeneralCourtNumber")
)

print(leg_count)


   GeneralCourtNumber  LegislatorCount
0                 183              161
1                 184              174
2                 185              176


In [130]:
matched_df.loc[
    matched_df["GeneralCourtNumber"] == 185,
    "RollCallName"
].unique()


array(['Mr. Speaker', 'Flanagan', 'Murphy, J.', 'Story', 'Alicea',
       'Flynn', 'Murphy, K.', 'Straus', 'Allen', 'Forry', 'Nangle',
       'Sullivan', 'Atkins', 'Fox', 'Natale', 'Swan', 'Atsalis',
       'Fresolo', 'Naughton', 'Timilty', 'Ayers', 'Galvin', 'Nyman',
       'Tobin', 'Balser', 'Garry', "O'Flaherty", 'Toomey', 'Binienda',
       'Gobi', 'Patrick', 'Torrisi', 'Bosley', 'Golden', 'Peake',
       'Turkington', 'Bradley', 'Grant', 'Pedone', 'Turner',
       'Brownsberger', 'Greene', 'Peisch', 'Vallee', 'Cabral', 'Guyer',
       'Petersen', 'Verga', 'Callahan', 'Haddad', 'Petrolati', 'Wagner',
       'Calter', 'Hall', 'Petruccelli', 'Wallace', 'Campbell', 'Harkins',
       'Pignatelli', 'Walrath', 'Canavan', 'Honan', 'Provost',
       'Walsh, M.', 'Canessa', 'Hynes', 'Puppolo', 'Walsh, S.', 'Casey',
       'Kafka', 'Quinn', 'Walz', 'Coakley-Rivera', 'Kane', 'Reinstein',
       'Welch', 'Conroy', 'Kaprielian', 'Rice', 'Wolf', 'Correia',
       'Kaufman', 'Richardson', 'Jones'

In [ ]:
invalid_names = [
    "Mr. Speaker",
    "ORDER EXTENSION - AMENDMENT",
    "REFERRING TO ECONOMIC DEVELOPMENT",
]

df_clean = matched_df[~matched_df["RollCallName"].isin(invalid_names)]

df_clean = df_clean.drop_duplicates()

df_clean = df_clean.reset_index(drop=True)

leg_count = (
    df_clean.groupby("GeneralCourtNumber")["RollCallName"]
        .nunique()
        .reset_index(name="UniqueLegislators")
        .sort_values("GeneralCourtNumber")
)
print(leg_count)

#df_clean.to_csv("rollcalls_183_185_cleaned.csv", index=False)

df_clean.head()

   GeneralCourtNumber  UniqueLegislators
0                 183                160
1                 184                172
2                 185                174


,GeneralCourtNumber,RollCallNumber,Date,TitleText,BillNumber,AmendmentNumber,Vote,RollCallName,YeasCount,NaysCount,NVCount,MatchedName,MatchedMemberCode,LeadershipPosition,Branch,District,Party,EmailAddress,RoomNumber,PhoneNumber
0,183,1,NaN,QUORUM,NaN,NaN,Y,Fresolo,151,0,9,John P. Fresolo,JPF1,NaN,House,NaN,Democrat,NaN,NaN,NaN
1,183,1,NaN,QUORUM,NaN,NaN,Y,"Murphy, K.J.",151,0,9,None,None,NaN,None,NaN,None,NaN,NaN,NaN
2,183,1,NaN,QUORUM,NaN,NaN,Y,Swan,151,0,9,Benjamin Swan,B S1,NaN,House,NaN,Democrat,NaN,NaN,NaN
3,183,1,NaN,QUORUM,NaN,NaN,Y,Asselin,151,0,9,Christopher P. Asselin,CPA1,NaN,House,NaN,Democrat,NaN,NaN,NaN
4,183,1,NaN,QUORUM,NaN,NaN,Y,Galvin,151,0,9,William C. Galvin,WCG1,NaN,House,NaN,Democrat,NaN,NaN,NaN


#### 2018-2025

In [ ]:
df_2018 = pd.read_csv("rollcalls_2018plus.csv")
matched_df_2018 = match_rollcall_to_members(df_2018, member_df)

df_2018_clean = matched_df_2018[~matched_df_2018["RollCallName"].isin(invalid_names)]

df_2018_clean = df_2018_clean.drop_duplicates()

# 重置索引
df_2108_clean = df_2018_clean.reset_index(drop=True)

leg_count = (
    df_2018_clean.groupby("GeneralCourtNumber")["RollCallName"]
        .nunique()
        .reset_index(name="UniqueLegislators")
        .sort_values("GeneralCourtNumber")
)
print(leg_count)

#df_clean.to_csv("rollcalls_183_185_cleaned.csv", index=False)

print(len(df_2018))
print(len(df_2018_clean))
print(len(df_2018) - len(df_2018_clean))

df_2018_clean.head()


Processing Session 190...
  Roll Call records: 31513, legislators list: 164
  Matched successfully: 143/158 unique names

Processing Session 191...
  Roll Call records: 94052, legislators list: 164
  Matched successfully: 150/163 unique names

Processing Session 192...
  Roll Call records: 43806, legislators list: 162
  Matched successfully: 150/160 unique names

Processing Session 193...
  Roll Call records: 31982, legislators list: 165
  Matched successfully: 150/165 unique names

Processing Session 194...
  Roll Call records: 16643, legislators list: 163
  Matched successfully: 147/163 unique names

UNKNOWN name (69):
  Session 190: Mr. Speaker
  Session 190: Walsh, C.
  Session 190: Walsh, T.
  Session 190: Williams
  Session 190: Moran M.
  Session 190: Moran F.
  Session 190: Hill
  Session 190: Gonzalez
  Session 190: Rogers, D.
  Session 190: Rogers, J.
  Session 190: Hunt, D.
  Session 190: Hunt, R.
  Session 190: Day
  Session 190: Mark
  Session 190: Stroke Program
  Sessio

,GeneralCourtNumber,RollCallNumber,Date,TitleText,BillNumber,AmendmentNumber,Vote,RollCallName,YeasCount,NaysCount,NVCount,MatchedName,MatchedMemberCode,LeadershipPosition,Branch,District,Party,EmailAddress,RoomNumber,PhoneNumber
1,190,306,01/24/2018 05:18 PM,QUORUM,NaN,NaN,X,Farley-Bouvier,148,0,11,Tricia Farley-Bouvier,TFB1,NaN,House,3rd Berkshire,Democrat,Tricia.Farley-Bouvier@mahouse.gov,156,617-722-2240
2,190,306,01/24/2018 05:18 PM,QUORUM,NaN,NaN,Y,McMurtry,148,0,11,Paul McMurtry,P_M1,NaN,House,11th Norfolk,Democrat,Paul.McMurtry@mahouse.gov,448,617-722-2582
3,190,306,01/24/2018 05:18 PM,QUORUM,NaN,NaN,Y,"Walsh, C.",148,0,11,None,None,None,None,None,None,None,None,None
4,190,306,01/24/2018 05:18 PM,QUORUM,NaN,NaN,Y,Mariano,148,0,11,Ronald Mariano,R_M1,Majority Leader,House,3rd Norfolk,Democrat,Ronald.Mariano@mahouse.gov,343,617-722-2300
5,190,306,01/24/2018 05:18 PM,QUORUM,NaN,NaN,Y,Fernandes,148,0,11,Dylan Fernandes,DAF1,NaN,House,"Barnstable, Dukes and Nantucket",Democrat,Dylan.Fernandes@mahouse.gov,236,617-722-2430


2009-2017

In [ ]:
df_2009 = pd.read_csv("house_rollcalls_2009_2017.csv")
df_2009 = df_2009.drop(columns=["Year"])
df_2009 = df_2009.rename(columns={"RollCallNumber": "RollcallNumber"})


matched_df_2009 = match_rollcall_to_members(df_2009, member_df)

df_2009_clean = matched_df_2009[~matched_df_2009["RollCallName"].isin(invalid_names)]

df_2009_clean = df_2009_clean.drop_duplicates()

df_2009_clean = df_2009_clean.reset_index(drop=True)

leg_count = (
    df_2009_clean.groupby("GeneralCourtNumber")["RollCallName"]
        .nunique()
        .reset_index(name="UniqueLegislators")
        .sort_values("GeneralCourtNumber")
)
print(leg_count)

print(len(df_2009))
print(len(df_2009_clean))
print(len(df_2009) - len(df_2009_clean))

df_2009_clean.head()


Processing Session 186...
  Roll Call records: 81247, legislators list: 162
  Matched successfully: 313/353 unique names

Processing Session 187...
  Roll Call records: 59091, legislators list: 164
  Matched successfully: 288/342 unique names

Processing Session 188...
  Roll Call records: 78844, legislators list: 169
  Matched successfully: 279/326 unique names

Processing Session 189...
  Roll Call records: 89229, legislators list: 166
  Matched successfully: 269/312 unique names

Processing Session 190...
  Roll Call records: 43347, legislators list: 164
  Matched successfully: 545/658 unique names

UNKNOWN name (297):
  Session 186: Mr. Speaker
  Session 186: Allen
  Session 186: Stanley, H.L.
  Session 186: Stanley, T.M.
  Session 186: Murphy, C.A.
  Session 186: Murphy, J.
  Session 186: Murphy, K.J.
  Session 186: Galvin
  Session 186: Bradley
  Session 186: Walsh, M.J.
  Session 186: Walsh, Ss.
  Session 186: Rush
  Session 186: Lewis
  Session 186: Walsh, S.
  Session 186: *A

,GeneralCourtNumber,RollCallNumber,Date,TitleText,BillNumber,AmendmentNumber,Vote,RollCallName,YeasCount,NaysCount,NVCount,MatchedName,MatchedMemberCode,LeadershipPosition,Branch,District,Party,EmailAddress,RoomNumber,PhoneNumber
0,186,1,"Wednesday, January 14, 2009",On adoption of an amendment (offered by Mr. Jo...,H.98,NaN,N,Ehrlich,20.0,135.0,5.0,Lori A. Ehrlich,LAE1,NaN,House,8th Essex,NaN,NaN,NaN,NaN
1,186,1,"Wednesday, January 14, 2009",On adoption of an amendment (offered by Mr. Jo...,H.98,NaN,N,Madden,20.0,135.0,5.0,Timothy R. Madden,TRM1,NaN,House,"Barnstable, Dukes and Nantucket",NaN,NaN,NaN,NaN
2,186,1,"Wednesday, January 14, 2009",On adoption of an amendment (offered by Mr. Jo...,H.98,NaN,N,Speliotis,20.0,135.0,5.0,Theodore C. Speliotis,TCS1,NaN,House,13th Essex,NaN,NaN,NaN,NaN
3,186,1,"Wednesday, January 14, 2009",On adoption of an amendment (offered by Mr. Jo...,H.98,NaN,N,Aguiar,20.0,135.0,5.0,Kevin Aguiar,K_A1,NaN,House,7th Bristol,NaN,NaN,NaN,NaN
4,186,1,"Wednesday, January 14, 2009",On adoption of an amendment (offered by Mr. Jo...,H.98,NaN,N,Fagan,20.0,135.0,5.0,James H. Fagan,JHF1,NaN,House,3rd Bristol,NaN,NaN,NaN,NaN


In [ ]:
df_early = df_clean   
df_middle = df_2009_clean                    # 183–185 
df_late = df_2018_clean                   # 190–194 


df_all = pd.concat([df_early, df_middle, df_late], ignore_index=True)

df_all = df_all.sort_values(
    by=["GeneralCourtNumber", "RollCallNumber"],
    ascending=[True, True]
).reset_index(drop=True)

df_all.to_csv("house_full_sessions_cleaned.csv", index=False)

print("final record count:", len(df_all))
df_all.head()


final record count: 788994


,GeneralCourtNumber,RollCallNumber,Date,TitleText,BillNumber,AmendmentNumber,Vote,RollCallName,YeasCount,NaysCount,NVCount,MatchedName,MatchedMemberCode,LeadershipPosition,Branch,District,Party,EmailAddress,RoomNumber,PhoneNumber
0,183,1,NaN,QUORUM,NaN,NaN,Y,Fresolo,151.0,0.0,9.0,John P. Fresolo,JPF1,NaN,House,NaN,Democrat,NaN,NaN,NaN
1,183,1,NaN,QUORUM,NaN,NaN,Y,"Murphy, K.J.",151.0,0.0,9.0,None,None,NaN,None,NaN,None,NaN,NaN,NaN
2,183,1,NaN,QUORUM,NaN,NaN,Y,Swan,151.0,0.0,9.0,Benjamin Swan,B S1,NaN,House,NaN,Democrat,NaN,NaN,NaN
3,183,1,NaN,QUORUM,NaN,NaN,Y,Asselin,151.0,0.0,9.0,Christopher P. Asselin,CPA1,NaN,House,NaN,Democrat,NaN,NaN,NaN
4,183,1,NaN,QUORUM,NaN,NaN,Y,Galvin,151.0,0.0,9.0,William C. Galvin,WCG1,NaN,House,NaN,Democrat,NaN,NaN,NaN


In [135]:
df_all_2 = df_all.drop(columns=["TitleText"])
df_all_2.to_csv("house_full_sessions_cleaned_2.csv", index=False)